# Treinamento e Exportação TinyML (ESP32) — Sistema de Irrigação Inteligente (PIBEX 2026 - IFPE Paulista)

Este notebook implementa o pipeline completo da **Fase 1** planejada em `docs/ACTION_PLAN_AI_FIRMWARE.md` e `docs/DATASETS_EVALUATION.md` para execução direta no **Google Colab** (ou localmente via Jupyter).

### Arquitetura e Decisões Técnicas Implementadas:
1. **Dataset Oficial:** *Smart Agriculture Dataset (`cropdata_updated.csv`)* com `16.411` registros.
2. **Classificação Multiclasse de 3 Estados (`Softmax`):**
   - **`0 (OFF)`:** Umidade adequada -> Válvula fechada.
   - **`1 (IRRIGATE)`:** Necessidade hídrica -> Válvula aberta.
   - **`2 (THERMAL_STRESS_WET_SOIL)`:** Calor extremo (>32 °C) com solo já úmido (acima da capacidade de campo) -> Mantém válvula **fechada** (evita asfixia radicular) e emite **Alerta de Estresse Térmico** no log CSV/painel.
3. **Contexto Agronômico Opcional (*Context Dropout* com Token `0 = UNKNOWN`):**
   - **Entradas Obrigatórias (Sensores ESP32):** Temperatura do Ar (`DHT22`), Umidade do Ar (`DHT22`) e Umidade do Solo (`HW-080`).
   - **Entradas Opcionais (Configuração do Canteiro no `LittleFS`):** `soil_type` (Tipo de Solo), `crop ID` (Cultura, priorizando hortaliças locais: Tomate, Pimenta/Pimentão, Cenoura e Batata) e `Seedling Stage` (Estágio de Crescimento).
   - **Execução Não-Bloqueante:** As variáveis opcionais são codificadas via *One-Hot Encoding* onde o estado `0 (UNKNOWN / Não Informado)` é representado pelo **vetor totalmente nulo (`0.0`)**. Durante o treino, aplicamos *Context Dropout* (mascarando aleatoriamente o contexto parcial ou total) para que a rede neural opere com **alta precisão tanto com contexto completo (~99-100%) quanto apenas com os 3 sensores físicos (sem nenhum contexto informado)**.
4. **Exportação Direta para o ESP32:** Gera `model.tflite` (quantizado), valida no `tf.lite.Interpreter`, extrai *Golden Test Vectors* e exporta o cabeçalho C/C++ `model_data.h` pronto para o PlatformIO.

## 1. Preparação do Ambiente e Dependências

In [ ]:
# No Google Colab, TensorFlow, NumPy, Pandas, Scikit-Learn e Matplotlib já vêm instalados.
import os
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
from sklearn.utils.class_weight import compute_class_weight

# Sementes fixas para total reprodutibilidade científica
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print(f"TensorFlow Version: {tf.__version__}")

## 2. Download e Carregamento Automático do Dataset (`kagglehub`)
Baixa automaticamente a versão mais recente do dataset **`chaitanyagopidesi/smart-agriculture-dataset`** diretamente do Kaggle via `kagglehub` (com fallback para o arquivo local do repositório caso já esteja baixado).

In [ ]:
import glob
import kagglehub

# Verifica primeiro se o CSV já existe localmente no repositório; caso contrário, baixa via kagglehub
LOCAL_PATHS = [
    "datasets/Smart Agriculture Dataset/cropdata_updated.csv",
    "ai-model/datasets/Smart Agriculture Dataset/cropdata_updated.csv",
    "cropdata_updated.csv",
]

dataset_path = next((p for p in LOCAL_PATHS if os.path.exists(p)), None)

if dataset_path is None:
    # Download latest version directly from Kaggle via kagglehub
    path = kagglehub.dataset_download("chaitanyagopidesi/smart-agriculture-dataset")
    print("Path to dataset files:", path)
    csv_files = glob.glob(os.path.join(path, "*.csv"))
    if not csv_files:
        raise FileNotFoundError(f"Nenhum arquivo CSV encontrado em: {path}")
    dataset_path = csv_files[0]

df = pd.read_csv(dataset_path)
print(f"Dataset carregado de: {dataset_path}")
print(f"Formato: {df.shape[0]} linhas x {df.shape[1]} colunas | Valores nulos: {df.isnull().sum().sum()}")
display(df.head())

## 3. Mapeamento Agronômico, Normalização e *Context Dropout* (Contexto Opcional)

Para que o firmware do **ESP32** nunca seja bloqueado caso o agricultor não informe o tipo de solo, a cultura ou o estágio da planta:
- Os **3 sensores físicos** são normalizados explicitamente no intervalo `[0.0, 1.0]` com limites físicos fixos exportados para o C++ (`TEMP: [0..50 °C]`, `HUMIDITY: [0..100 %]`, `SOIL_MOISTURE: [0..100 %]`).
- As **3 variáveis de contexto** (`soil_type`, `crop ID`, `Seedling Stage`) têm o código `0` reservado para **`UNKNOWN` (Não Informado)** e são convertidas em vetores *One-Hot* onde `0 = UNKNOWN` gera um vetor de zeros `[0, 0, ..., 0]`.
- Assim, o vetor de entrada do modelo possui **23 features float32**:
  - Índices `0..2` (3): `[temp_norm, humidity_norm, soil_moisture_norm]`
  - Índices `3..9` (7): *One-Hot* `soil_type` (se `UNKNOWN`, todos `0.0`)
  - Índices `10..14` (5): *One-Hot* `crop ID` (se `UNKNOWN`, todos `0.0`)
  - Índices `15..22` (8): *One-Hot* `Seedling Stage` (se `UNKNOWN`, todos `0.0`)

In [ ]:
# Limites explícitos de normalização compartilhados entre Python e ESP32 (C++)
TEMP_MIN, TEMP_MAX = 0.0, 50.0
HUM_MIN, HUM_MAX = 0.0, 100.0
SOIL_MIN, SOIL_MAX = 0.0, 100.0

# Dicionários oficiais de contexto (Índice 0 é sempre reservado para UNKNOWN / Não Informado)
SOIL_TYPES = [
    "Clay Soil", "Sandy Soil", "Red Soil", "Loam Soil",
    "Black Soil", "Alluvial Soil", "Chalky Soil"
]
CROP_TYPES = [
    "Tomato", "Chilli", "Carrot", "Potato", "Wheat"
]
SEEDLING_STAGES = [
    "Germination", "Seedling Stage",
    "Vegetative Growth / Root or Tuber Development",
    "Flowering", "Pollination", "Fruit/Grain/Bulb Formation",
    "Maturation", "Harvest"
]

SOIL_TO_ID = {name: i + 1 for i, name in enumerate(SOIL_TYPES)}     # 1..7 (0 = UNKNOWN)
CROP_TO_ID = {name: i + 1 for i, name in enumerate(CROP_TYPES)}     # 1..5 (0 = UNKNOWN)
STAGE_TO_ID = {name: i + 1 for i, name in enumerate(SEEDLING_STAGES)} # 1..8 (0 = UNKNOWN)

INPUT_DIM = 3 + len(SOIL_TYPES) + len(CROP_TYPES) + len(SEEDLING_STAGES) # 3 + 7 + 5 + 8 = 23
NUM_CLASSES = 3 # 0: OFF, 1: IRRIGATE, 2: THERMAL_STRESS_WET_SOIL

def encode_features(temp, hum, moi, soil_id=0, crop_id=0, stage_id=0):
    """
    Codifica as 3 leituras dos sensores + 3 IDs de contexto opcionais (0 = UNKNOWN)
    em um vetor float32 de 23 posições idêntico ao que será montado no ESP32.
    """
    vec = np.zeros(INPUT_DIM, dtype=np.float32)
    vec[0] = np.clip((temp - TEMP_MIN) / (TEMP_MAX - TEMP_MIN), 0.0, 1.0)
    vec[1] = np.clip((hum - HUM_MIN) / (HUM_MAX - HUM_MIN), 0.0, 1.0)
    vec[2] = np.clip((moi - SOIL_MIN) / (SOIL_MAX - SOIL_MIN), 0.0, 1.0)

    if 1 <= soil_id <= len(SOIL_TYPES):
        vec[3 + (soil_id - 1)] = 1.0
    if 1 <= crop_id <= len(CROP_TYPES):
        vec[3 + len(SOIL_TYPES) + (crop_id - 1)] = 1.0
    if 1 <= stage_id <= len(SEEDLING_STAGES):
        vec[3 + len(SOIL_TYPES) + len(CROP_TYPES) + (stage_id - 1)] = 1.0
    return vec

print(f"Dimensão total do vetor de entrada: {INPUT_DIM} features")

## 4. Divisão Estratificada (70% Treino / 15% Validação / 15% Teste) e Data Augmentation com *Context Dropout*

1. Primeiro dividimos as `16.411` amostras reais em **Treino (70%)**, **Validação (15%)** e **Teste (15%)** de forma estratificada por `crop ID` e `result` (garantindo zero vazamento de dados / *zero data leakage*).
2. No conjunto de **Treino**, aplicamos *Context Dropout* gerando variações de cada amostra:
   - **Contexto Completo:** `(temp, hum, moi, soil_id, crop_id, stage_id)`
   - **Sem Estágio Fenológico:** `(temp, hum, moi, soil_id, crop_id, 0)` (o agricultor só configurou solo e cultura ao plantar)
   - **Apenas Tipo de Solo:** `(temp, hum, moi, soil_id, 0, 0)`
   - **Sem Nenhum Contexto (Só os 3 Sensores Físicos):** `(temp, hum, moi, 0, 0, 0)` — Aplicado exclusivamente às amostras de **Hortaliças (`Tomato`, `Chilli`, `Carrot`, `Potato`)**, garantindo que quando o agricultor não informar a cultura, a rede neural assuma o comportamento protetor das hortaliças locais de Paulista-PE em vez do Trigo (`Wheat`).

In [ ]:
df["soil_id"] = df["soil_type"].map(SOIL_TO_ID)
df["crop_id"] = df["crop ID"].map(CROP_TO_ID)
df["stage_id"] = df["Seedling Stage"].map(STAGE_TO_ID)
df["strat_key"] = df["crop ID"].astype(str) + "_" + df["result"].astype(str)

train_df, temp_df = train_test_split(
    df, test_size=0.30, random_state=SEED, stratify=df["strat_key"]
)
val_df, test_df = train_test_split(
    temp_df, test_size=0.50, random_state=SEED, stratify=temp_df["strat_key"]
)

print(f"Amostras Reais -> Treino: {len(train_df)} | Validação: {len(val_df)} | Teste: {len(test_df)}")

def build_augmented_dataset(dataframe, is_training=True):
    X_list, y_list = [], []
    for _, row in dataframe.iterrows():
        t, h, m = float(row["temp"]), float(row["humidity"]), float(row["MOI"])
        s_id, c_id, st_id = int(row["soil_id"]), int(row["crop_id"]), int(row["stage_id"])
        label = int(row["result"])
        is_vegetable = (row["crop ID"] != "Wheat")

        # 1. Contexto completo sempre incluído
        X_list.append(encode_features(t, h, m, s_id, c_id, st_id))
        y_list.append(label)

        if is_training:
            # 2. Contexto estático do canteiro (Solo + Cultura, sem Estágio)
            X_list.append(encode_features(t, h, m, s_id, c_id, 0))
            y_list.append(label)

            # 3. Apenas Solo informado (para hortaliças)
            if is_vegetable:
                X_list.append(encode_features(t, h, m, s_id, 0, 0))
                y_list.append(label)

                # 4. Nenhum contexto informado (0, 0, 0 -> Fallback para Hortaliças Locais)
                # Para evitar ambiguidade na Classe 2 sem solo informado, mantemos 0 e 1 puros
                # e amostras de Classe 2 com MOI claramente elevado (>= 60%)
                if label in (0, 1) or (label == 2 and m >= 60.0):
                    X_list.append(encode_features(t, h, m, 0, 0, 0))
                    y_list.append(label)

    return np.array(X_list, dtype=np.float32), np.array(y_list, dtype=np.int32)

X_train, y_train = build_augmented_dataset(train_df, is_training=True)
X_val, y_val = build_augmented_dataset(val_df, is_training=True)

# Pesos de classe balanceados para garantir alto Recall na Classe 2 (Alerta de Estresse Térmico)
classes_unique = np.unique(y_train)
weights = compute_class_weight(class_weight="balanced", classes=classes_unique, y=y_train)
class_weight_dict = {int(c): float(w) for c, w in zip(classes_unique, weights)}

print(f"Treino após Context Dropout: {X_train.shape[0]} amostras | Validação: {X_val.shape[0]} amostras")
print(f"Pesos de Classe (Class Weights): {class_weight_dict}")

## 5. Arquitetura da Rede Neural MLP (TinyML) e Treinamento

A arquitetura utiliza apenas camadas `Dense` com ativação `ReLU` e camada final `Softmax` de 3 neurônios (`AddFullyConnected`, `AddRelu`, `AddSoftmax` no TensorFlow Lite Micro), ocupando menos de **8 KB de Flash** e **4 KB de `tensor_arena` (RAM)** no ESP32.

In [ ]:
def build_tinyml_model(input_dim=INPUT_DIM, num_classes=NUM_CLASSES):
    model = tf.keras.Sequential([
        tf.keras.layers.Input(shape=(input_dim,), name="sensor_and_context_input"),
        tf.keras.layers.Dense(32, activation="relu", name="dense_1"),
        tf.keras.layers.Dense(16, activation="relu", name="dense_2"),
        tf.keras.layers.Dense(num_classes, activation="softmax", name="decision_output")
    ])
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=0.003),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )
    return model

model = build_tinyml_model()
model.summary()

callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss", patience=12, restore_best_weights=True, verbose=1
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss", factor=0.5, patience=5, min_lr=1e-5, verbose=1
    )
]

history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=80,
    batch_size=64,
    class_weight=class_weight_dict,
    callbacks=callbacks,
    verbose=1
)

## 6. Avaliação em Múltiplos Cenários de Operação do ESP32

Avaliamos o mesmo modelo único nos **4 modos de uso reais em campo** sobre o conjunto de **Teste (15% nunca vistos)**:
1. **Modo 1 — Contexto Completo:** Usuário informou Solo + Cultura + Estágio.
2. **Modo 2 — Canteiro Fixo:** Usuário informou apenas Solo + Cultura (`stage = UNKNOWN`).
3. **Modo 3 — Apenas Solo:** Usuário informou apenas o Tipo de Solo (`crop = UNKNOWN`, `stage = UNKNOWN`).
4. **Modo 4 — Zero Contexto (Plug & Play):** Usuário **não informou nada** (`soil = 0, crop = 0, stage = 0`), operando 100% autônomo apenas com o DHT22 e o HW-080 nas hortaliças.

In [ ]:
CLASS_NAMES = ["0: OFF (Nao Irrigar)", "1: IRRIGATE (Irrigar)", "2: THERMAL_STRESS (Alerta)"]

def evaluate_scenario(dataframe, mode_name, mask_fn):
    X_s, y_s = [], []
    for _, row in dataframe.iterrows():
        s_id, c_id, st_id = mask_fn(int(row["soil_id"]), int(row["crop_id"]), int(row["stage_id"]))
        X_s.append(encode_features(float(row["temp"]), float(row["humidity"]), float(row["MOI"]), s_id, c_id, st_id))
        y_s.append(int(row["result"]))
    X_s, y_s = np.array(X_s, dtype=np.float32), np.array(y_s, dtype=np.int32)
    preds = np.argmax(model.predict(X_s, verbose=0), axis=1)
    print(f"\n{=*75}\nAvaliação: {mode_name} (n={len(y_s)})\n{=*75}")
    print(classification_report(y_s, preds, target_names=CLASS_NAMES, digits=4))
    return y_s, preds

# 1. Contexto Completo (Todas as culturas)
y_true_full, y_pred_full = evaluate_scenario(
    test_df, "MODO 1: Contexto Completo (Sensores + Solo + Cultura + Estágio)",
    lambda s, c, st: (s, c, st)
)

# 2. Solo + Cultura (Sem estágio)
_ = evaluate_scenario(
    test_df, "MODO 2: Canteiro Fixo (Sensores + Solo + Cultura | Estágio = UNKNOWN)",
    lambda s, c, st: (s, c, 0)
)

# 3. Apenas Solo (Hortaliças)
veg_test_df = test_df[test_df["crop ID"] != "Wheat"]
_ = evaluate_scenario(
    veg_test_df, "MODO 3: Apenas Solo Informado (Hortaliças | Cultura e Estágio = UNKNOWN)",
    lambda s, c, st: (s, 0, 0)
)

# 4. Zero Contexto - Apenas 3 Sensores Físicos (Hortaliças)
y_true_zero, y_pred_zero = evaluate_scenario(
    veg_test_df, "MODO 4: Zero Contexto / Só 3 Sensores DHT22+HW080 (Hortaliças | Tudo = UNKNOWN)",
    lambda s, c, st: (0, 0, 0)
)

# Plotagem das Curvas de Aprendizado e Matriz de Confusão
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
axes[0].plot(history.history["loss"], label="Treino Loss")
axes[0].plot(history.history["val_loss"], label="Validação Loss")
axes[0].set_title("Evolução da Loss (Crossentropy)")
axes[0].legend()

ConfusionMatrixDisplay.from_predictions(
    y_true_full, y_pred_full, display_labels=["OFF", "IRRIG", "STRESS"], ax=axes[1], cmap="Greens"
)
axes[1].set_title("Matriz de Confusão (Contexto Completo)")

ConfusionMatrixDisplay.from_predictions(
    y_true_zero, y_pred_zero, display_labels=["OFF", "IRRIG", "STRESS"], ax=axes[2], cmap="Blues"
)
axes[2].set_title("Matriz de Confusão (Só 3 Sensores - Sem Contexto)")
plt.tight_layout()
plt.show()

## 7. Conversão para TensorFlow Lite (`.tflite`), Validação Cruzada e Exportação de `model_data.h` para o ESP32

Esta célula:
1. Converte o modelo Keras para `.tflite` com otimizações para TensorFlow Lite Micro.
2. Executa o `tf.lite.Interpreter` em Python comparando todas as saídas contra o modelo Keras original.
3. Gera os **Golden Test Vectors** (casos reais de teste para as classes `0`, `1` e `2`, com e sem contexto) e grava o arquivo `model_data.h` pronto para ser incluído em `embedded/include/model_data.h`.

In [ ]:
OUTPUT_DIR = "output"
os.makedirs(OUTPUT_DIR, exist_ok=True)
tflite_path = os.path.join(OUTPUT_DIR, "model.tflite")
header_path = os.path.join(OUTPUT_DIR, "model_data.h")

# Conversão para TFLite (Float32 / Otimizado para TFLite Micro sem perda em Softmax multiclasse)
converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
tflite_model = converter.convert()

with open(tflite_path, "wb") as f:
    f.write(tflite_model)

print(f"[TinyML] Arquivo salvo: {tflite_path} ({len(tflite_model)} bytes / {len(tflite_model)/1024:.2f} KB)")

# Validação cruzada Keras vs TFLite Interpreter
interpreter = tf.lite.Interpreter(model_content=tflite_model)
interpreter.allocate_tensors()
input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()

def predict_tflite(x_vec):
    interpreter.set_tensor(input_details[0]["index"], np.expand_dims(x_vec, axis=0))
    interpreter.invoke()
    return interpreter.get_tensor(output_details[0]["index"])[0]

max_diff = 0.0
for i in range(min(200, len(X_val))):
    p_keras = model.predict(np.expand_dims(X_val[i], axis=0), verbose=0)[0]
    p_tflite = predict_tflite(X_val[i])
    max_diff = max(max_diff, float(np.max(np.abs(p_keras - p_tflite))))

print(f"[Validação Cruzada] Diferença máxima absoluta Keras vs TFLite em 200 amostras: {max_diff:.6f}")

# Geração de Golden Test Vectors para validar no Serial Monitor do ESP32
golden_cases = [
    ("Caso 1 (Sem Contexto - Solo Umido/Frio -> OFF)", 22.0, 78.0, 65.0, 0, 0, 0),
    ("Caso 2 (Sem Contexto - Solo Seco/Quente -> IRRIGATE)", 34.0, 45.0, 22.0, 0, 0, 0),
    ("Caso 3 (Sem Contexto - Calor Extremo + Solo Encharcado -> THERMAL_STRESS)", 38.0, 40.0, 75.0, 0, 0, 0),
    ("Caso 4 (Com Contexto: Tomate em Solo Argiloso + Floracao - Seco -> IRRIGATE)", 33.0, 50.0, 28.0, SOIL_TO_ID["Clay Soil"], CROP_TO_ID["Tomato"], STAGE_TO_ID["Flowering"]),
    ("Caso 5 (Com Contexto: Pimenta em Solo Arenoso + Germinacao - Calor + Umido -> THERMAL_STRESS)", 37.0, 42.0, 70.0, SOIL_TO_ID["Sandy Soil"], CROP_TO_ID["Chilli"], STAGE_TO_ID["Germination"]),
]

golden_comments = []
print("\n--- GOLDEN TEST VECTORS (TFLite Output) ---")
for desc, t, h, m, s_id, c_id, st_id in golden_cases:
    vec = encode_features(t, h, m, s_id, c_id, st_id)
    probs = predict_tflite(vec)
    pred_cls = int(np.argmax(probs))
    line = f"// {desc}: Temp={t}C, Hum={h}%, Soil={m}%, SoilID={s_id}, CropID={c_id}, StageID={st_id} => Class={pred_cls} (P0={probs[0]:.3f}, P1={probs[1]:.3f}, P2={probs[2]:.3f})"
    golden_comments.append(line)
    print(line)

# Formata array hexadecimal C/C++
hex_lines = []
for i in range(0, len(tflite_model), 12):
    chunk = ", ".join(f"0x{b:02x}" for b in tflite_model[i:i+12])
    hex_lines.append(f"  {chunk}")
hex_body = ",\n".join(hex_lines)
golden_block = "\n".join(golden_comments)

c_header = f"""#pragma once

#include <cstdint>

// ============================================================================
// Modelo TinyML Gerado Automaticamente - Sistema de Irrigacao Inteligente
// Arquitetura: MLP ({INPUT_DIM} inputs -> 32 ReLU -> 16 ReLU -> 3 Softmax)
// ============================================================================
// Golden Test Vectors para validacao no ESP32:
{golden_block}
// ============================================================================

namespace TinyMLConfig {{
    constexpr uint8_t INPUT_FEATURE_COUNT = {INPUT_DIM};
    constexpr uint8_t OUTPUT_CLASS_COUNT = {NUM_CLASSES};

    constexpr float TEMP_MIN = {TEMP_MIN}f;
    constexpr float TEMP_MAX = {TEMP_MAX}f;
    constexpr float HUMIDITY_MIN = {HUM_MIN}f;
    constexpr float HUMIDITY_MAX = {HUM_MAX}f;
    constexpr float SOIL_MIN = {SOIL_MIN}f;
    constexpr float SOIL_MAX = {SOIL_MAX}f;

    constexpr float DEFAULT_CONFIDENCE_THRESHOLD = 0.55f;
    constexpr uint8_t THERMAL_STRESS_SOIL_CAP_PCT = 65;
    constexpr int8_t THERMAL_STRESS_TEMP_MIN_C = 32;

    enum class SoilType : uint8_t {{
        UNKNOWN = 0,
        CLAY_SOIL = 1,
        SANDY_SOIL = 2,
        RED_SOIL = 3,
        LOAM_SOIL = 4,
        BLACK_SOIL = 5,
        ALLUVIAL_SOIL = 6,
        CHALKY_SOIL = 7
    }};

    enum class CropType : uint8_t {{
        UNKNOWN = 0, // Fallback para Hortalicas Locais
        TOMATO = 1,
        CHILLI = 2,
        CARROT = 3,
        POTATO = 4,
        WHEAT = 5
    }};

    enum class SeedlingStage : uint8_t {{
        UNKNOWN = 0,
        GERMINATION = 1,
        SEEDLING_STAGE = 2,
        VEGETATIVE_GROWTH = 3,
        FLOWERING = 4,
        POLLINATION = 5,
        FRUIT_FORMATION = 6,
        MATURATION = 7,
        HARVEST = 8
    }};
}}

const unsigned int g_model_len = {len(tflite_model)};
alignas(8) const unsigned char g_model_data[] = {{
{hex_body}
}};
"""

with open(header_path, "w", encoding="utf-8") as f:
    f.write(c_header)

print(f"\n[TinyML] Header C++ exportado com sucesso em: {header_path}")

# Se estiver rodando localmente dentro do repositório, também copia para embedded/include/model_data.h
for emb_dir in ["../embedded/include", "embedded/include"]:
    if os.path.isdir(emb_dir):
        dest = os.path.join(emb_dir, "model_data.h")
        with open(dest, "w", encoding="utf-8") as f:
            f.write(c_header)
        print(f"[TinyML] Header sincronizado automaticamente com o firmware: {dest}")

# Se estiver no Google Colab, oferece download imediato do model_data.h
try:
    from google.colab import files
    files.download(header_path)
except ImportError:
    pass